In [1]:
%load_ext autoreload
%autoreload 2

import os, pandas as pd, numpy as np
import requests
from dotenv import load_dotenv
from collections import defaultdict
load_dotenv(override=False)

from etl.core.db import PortXDB
db = PortXDB()

In [2]:
db.ping()

('PortX', 'postgres', IPv4Address('127.0.0.1'), 5432)

In [11]:
print("Ping:", db.ping())

Ping: ('PortX', 'postgres', IPv6Address('::1'), 5432)


In [15]:
db.fetch_all("SELECT * FROM public.country.py")

[(1,
  'Pakistan',
  'PK',
  'PKR',
  'Pakistani Rupee',
  '₨',
  'Asia',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 (2,
  'United Arab Emirates',
  'AE',
  'AED',
  'UAE Dirham',
  'د.إ',
  'Middle East',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 (3,
  'Saudi Arabia',
  'SA',
  'SAR',
  'Saudi Riyal',
  '﷼',
  'Middle East',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 (4,
  'Qatar',
  'QA',
  'QAR',
  'Qatari Riyal',
  'ر.ق',
  'Middle East',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 (5,
  'Kuwait',
  'KW',
  'KWD',
  'Kuwaiti Dinar',
  'د.ك',
  'Middle East',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 (6,
  'Bahrain',
  'BH',
  'BHD',
  'Bahraini Dinar',
  '.د.ب',
  'Middle East',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 (7,
  'Oman',
  'OM',
  'OMR',
  'Omani Rial',
  'ر.ع.',
  'Middle East',
  True,
  datetime.datetime(2025, 9, 22, 16, 59, 51, 966078)),
 

In [14]:
rows = db.fetch_all(
    """
    select table_schema, table_name
    from information_schema.tables
    where table_type = 'BASE TABLE'
      and table_schema not in ('pg_catalog', 'information_schema')
    order by table_schema, table_name;
    """
)
for schema, table in rows:
    print(f"{schema}.{table}")

portx.equity_attributes
portx.fx_rate_daily
portx.issuer
portx.ref_asset_class
portx.ref_currency
portx.ref_currency_pair
portx.ref_security_type
portx.ref_sub_asset_class
portx.security_gics_classification
portx.security_identifier
portx.security_listing
portx.security_master
public.country
public.exchange
public.map_psx_gics
public.ref_asset_class
public.ref_gics_edition
public.ref_gics_industries
public.ref_gics_industry_groups
public.ref_gics_sectors
public.ref_gics_subindustries
public.ref_psx_sectors
public.ref_security_type
public.ref_sub_asset_class


In [9]:
currency_pairs_codes = db.fetch_all("select * from portx.ref_currency_pair")
for code in currency_pairs_codes:
    print(code[0],code[3] ) 

1 EUR/USD
2 USD/EUR
3 USD/PKR
4 EUR/PKR
5 USD/AED
6 EUR/AED
7 USD/SAR
8 EUR/SAR
9 USD/JPY
10 EUR/JPY


In [6]:
currency_pair_list  = []


currency_pairs_codes = db.fetch_all("select * from portx.ref_currency_pair")
for code in currency_pairs_codes:
    currency_pair_list.append((code[0], code[3]))  

In [7]:
currency_pair_list

[(1, 'EUR/USD'),
 (2, 'USD/EUR'),
 (3, 'USD/PKR'),
 (4, 'EUR/PKR'),
 (5, 'USD/AED'),
 (6, 'EUR/AED'),
 (7, 'USD/SAR'),
 (8, 'EUR/SAR'),
 (9, 'USD/JPY'),
 (10, 'EUR/JPY')]

In [8]:
# fx_read_pairs_only.py
from __future__ import annotations
from typing import Dict, List, Tuple, Iterable
import os, requests

from etl.core.db import PortXDB  # uses your .env for DB creds

EXR_KEY = os.getenv("EXCHANGERATE_API_KEY")  # put in .env

class FXError(RuntimeError): ...

def _exr_fetch(base: str, targets: Iterable[str]) -> Dict[str, float]:
    """Call ExchangeRate-API once for a base, return {QUOTE: rate}."""
    if not EXR_KEY:
        raise FXError("Set EXCHANGERATE_API_KEY in your .env")
    url = f"https://v6.exchangerate-api.com/v6/{EXR_KEY}/latest/{base}"
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    j = resp.json()
    if j.get("result") != "success":
        raise FXError(f"ExchangeRate-API error: {j}")
    table = j.get("conversion_rates", {})
    return {q: float(table[q]) for q in targets if q in table}

def read_pair_rates(db: PortXDB) -> List[Tuple[int, str, float]]:
    """
    Pull pairs from DB and fetch rates strictly from ExchangeRate-API.
    Returns: [(pair_id, "BASE/QUOTE", rate_target_per_1_base)]
    """
    pairs = db.fetch_all("""
        SELECT currency_pair_id, base_ccy, quote_ccy
        FROM portx.ref_currency_pair
        WHERE COALESCE(is_active, true) = true
        ORDER BY currency_pair_id
    """)

    # group by base to minimize API calls
    by_base: Dict[str, List[Tuple[int, str]]] = {}
    for pair_id, base, quote in pairs:
        by_base.setdefault(base, []).append((pair_id, quote))

    out: List[Tuple[int, str, float]] = []
    for base, items in by_base.items():
        targets = [q for _, q in items]
        rates = _exr_fetch(base, targets)  # single provider only∂
        for pair_id, quote in items:
            r = rates.get(quote)
            if r is not None:
                out.append((pair_id, f"{base}/{quote}", r))
            # if you prefer to fail when missing, raise instead:
            # else: raise FXError(f"Missing {base}/{quote} from provider")

    return out

# --- Example usage (in a script or notebook cell) ---
if __name__ == "__main__":
    db = PortXDB()
    data = read_pair_rates(db)
    print("Pairs & rates:")
    for row in data:
        print(row)
    # data is now like: [(1, 'EUR/USD', 1.0743), (2, 'USD/EUR', 0.931), ...]


UndefinedColumn: column "base_ccy" does not exist
LINE 2:         SELECT currency_pair_id, base_ccy, quote_ccy
                                         ^